<a href="https://colab.research.google.com/github/NicoMontal/daiist-env-setup/blob/main/pytorch_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch, Hands-On

A trained model that only exists in a notebook creates no value for anyone
outside the room. Today we close part of that gap ourselves. This whole
session is this one notebook: we'll build up PyTorch from the ground up,
covering tensors, the operations you'll use constantly, and autograd. We'll
finish by training a **linear regression model with several input
features**, from scratch, on data we generate ourselves. Because we
generate the data, we know the *true* relationship, so we can check whether
training actually recovered it.

This is exactly the muscle Assignment 1 asks you to use, on a real dataset
you pick yourself. The one difference is that there you'll use the
standard, convenient PyTorch API (`torch.optim`) instead of writing the
update rule by hand like we do today. Today is about understanding what
that convenience hides. The appendix at the very end shows that convenient
version worked out in full.

Run every cell as we go. There's nothing to install beyond the environment
you already have working. Cells marked **Try it** are for you to fill in
during class and submit your notebook on Blackboard for class participation.

**DO NOT USE AI TO COMPLETE THIS NOTEBOOK**


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import torch

print("torch version:", torch.__version__)


torch version: 2.2.2


## 1. Tensors: PyTorch's core object

A `torch.Tensor` is an n-dimensional array, like a NumPy array. Same idea,
different library, with one superpower we get to further down: it can
track its own gradient.

### 1.1 Creating tensors


In [3]:
# From a Python list
a = torch.tensor([1.0, 2.0, 3.0])
print(a)
print("dtype:", a.dtype)
print("shape:", a.shape)
print("ndim: ", a.ndim)

tensor([1., 2., 3.])
dtype: torch.float32
shape: torch.Size([3])
ndim:  1


In [4]:
# 2D: a matrix
m = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
print(m)
print("shape:", m.shape)   # (rows, columns) -- (3, 2) here


tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
shape: torch.Size([3, 2])


In [5]:
# Convenience constructors you'll use constantly
print(torch.zeros(3))
print(torch.ones(2, 3))
print(torch.arange(0, 10, 2))
print(torch.eye(3))          # identity matrix


tensor([0., 0., 0.])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([0, 2, 4, 6, 8])
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])


### 1.2 Dtypes matter

PyTorch is strict about dtypes in a way NumPy is more relaxed about. Mixing
`float32` and `float64` (or `float` and `int`) in the same operation is a
common source of errors. Almost everything we do today uses `float32`: it's
PyTorch's default for `torch.tensor(...)` when you pass floats, and it's
the standard dtype for training.


In [6]:
int_tensor = torch.tensor([1, 2, 3])          # dtype inferred as int64
float_tensor = torch.tensor([1.0, 2.0, 3.0])  # dtype inferred as float32
print(int_tensor.dtype, float_tensor.dtype)

# Convert explicitly when you need to
converted = int_tensor.to(torch.float32)
print(converted.dtype)


torch.int64 torch.float32
torch.float32


**Try it**: `float_tensor` above is `float32`. Create `double_tensor`,
holding the same values as `float64`, and print its dtype to confirm.


In [11]:
# TODO: create double_tensor, a float64 version of float_tensor
double_tensor = float_tensor.to(torch.float64)
print(double_tensor.dtype)

torch.float64


## 2. Indexing, slicing, reshaping

Same rules as NumPy, if you've used it before.


In [12]:
x = torch.arange(12).reshape(3, 4).float()
print(x)
print("row 0:       ", x[0])
print("column 1:    ", x[:, 1])
print("bottom-right:", x[1:, 2:])


tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
row 0:        tensor([0., 1., 2., 3.])
column 1:     tensor([1., 5., 9.])
bottom-right: tensor([[ 6.,  7.],
        [10., 11.]])


In [13]:
# Reshaping: total number of elements must stay the same
print(x.shape)
print(x.reshape(4, 3).shape)
print(x.reshape(-1).shape)      # -1 means "infer this dimension" -- flatten

# unsqueeze/squeeze add or remove a size-1 dimension -- you'll see this
# constantly when a function expects a batch dimension you don't have yet
v = torch.tensor([1.0, 2.0, 3.0])
print(v.shape, v.unsqueeze(0).shape, v.unsqueeze(1).shape)


torch.Size([3, 4])
torch.Size([4, 3])
torch.Size([12])
torch.Size([3]) torch.Size([1, 3]) torch.Size([3, 1])


**Try it**: using `x` above (shape `(3, 4)`), extract columns 0 and 2 only,
in a single indexing expression, and print the result's shape.


In [24]:
print(x[:,0::2])

tensor([[ 0.,  2.],
        [ 4.,  6.],
        [ 8., 10.]])


## 3. Elementwise operations and broadcasting

Arithmetic on tensors is elementwise by default, and PyTorch **broadcasts**
smaller tensors up to match larger ones the same way NumPy does. This is
what lets us add a single bias term to every row of a batch, further down.


In [25]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print(a + b)
print(a * b)          # elementwise, NOT matrix multiplication
print(a ** 2)


tensor([11., 22., 33.])
tensor([10., 40., 90.])
tensor([1., 4., 9.])


In [26]:
# Broadcasting: a (3, 2) matrix plus a (2,) vector -- the vector is applied
# to every row
batch = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
bias = torch.tensor([100.0, 1000.0])
print(batch + bias)


tensor([[ 101., 1002.],
        [ 103., 1004.],
        [ 105., 1006.]])


**Try it**: compute the elementwise average of `a` and `b` from above, call
it `avg`, using only arithmetic operators (no `.mean()` on a stacked
tensor).


In [33]:
# TODO: avg = ...
avg = (a+b) / 2
print(avg)


tensor([ 5.5000, 11.0000, 16.5000])


## 4. Matrix multiplication: the operation our model needs

`*` is elementwise. Matrix multiplication is a different operator: `@` (or
`torch.matmul`). This is the single most important operation for today: it's
how a model combines *several* input features into one prediction.


In [34]:
X = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])   # shape (2 samples, 3 features)
w = torch.tensor([10.0, 0.0, 1.0])    # shape (3 features,) -- one weight per feature

# For each row of X, multiply elementwise by w and sum -- that's exactly
# what matrix-vector multiplication does:
print(X @ w)          # shape (2,) -- one prediction per sample
print(X @ w + 5.0)    # add a scalar bias, broadcast to every sample


tensor([13., 46.])
tensor([18., 51.])


Compare `X @ w` to doing it by hand for row 0: `1*10 + 2*0 + 3*1 = 13`. That
matches `(X @ w)[0]` above. Every model we train today, and later in this
course, computes something built out of this operation.

**Try it**: define a new weight vector `w2` where every entry is `1.0`.
Before running, predict what `X @ w2` will be. Then compute it and check
your prediction.


In [35]:
# TODO: define w2 and compute X @ w2
w2 = torch.tensor([1,1,1]).float()

X @ w2

tensor([ 6., 15.])

## 5. Reductions: sum, mean

Used constantly to turn a vector of per-sample errors into one scalar loss.


In [36]:
errors = torch.tensor([4.0, 9.0, 1.0, 0.0])
print("sum: ", errors.sum())
print("mean:", errors.mean())

# .item() pulls a single-element tensor out as a plain Python float --
# useful for printing/logging, not for anything you'll keep computing with
print(errors.mean().item(), type(errors.mean().item()))


sum:  tensor(14.)
mean: tensor(3.5000)
3.5 <class 'float'>


**Try it**: compute the sum of squared errors (the numerator of MSE, before
dividing by the count) for `errors` above, in one line.


In [41]:
# TODO: sse = ...
sse = (errors ** 2).sum()
print(sse)

tensor(98.)


## 6. Bridging with NumPy

If you already know NumPy, converting between the two is cheap and common:
for example, loading data with pandas or NumPy, then handing it to PyTorch
for training.


In [46]:
np_array = np.array([1.0, 2.0, 3.0])
as_tensor = torch.from_numpy(np_array).float()
back_to_numpy = as_tensor.numpy()

print(type(np_array), type(as_tensor), type(back_to_numpy))


<class 'numpy.ndarray'> <class 'torch.Tensor'> <class 'numpy.ndarray'>


**Try it**: create a small NumPy array of your choosing, convert it to a
PyTorch tensor, multiply it by 10, then convert the result back to NumPy.


In [47]:
# TODO: create a NumPy array, convert to a tensor, multiply by 10, convert
# it back to NumPy

array = np.array([1.0, 6.0, 9.0])
tensor = torch.from_numpy(array).float()
back_to_numpy = (tensor * 10).numpy()
print(back_to_numpy)

[10. 60. 90.]


## 7. Autograd: the feature that makes training possible

Everything above also exists in NumPy. Here's what's different about
PyTorch: if you create a tensor with `requires_grad=True`, PyTorch starts
recording every operation it participates in, building a **computation
graph**. Call `.backward()` on a final scalar (almost always a loss), and
PyTorch walks that graph backwards, computing the gradient of the loss with
respect to every tensor that asked for one, via the chain rule. No calculus
written by you.


In [61]:
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)                      # data -- doesn't need a gradient

y = w * x          # y = 6.0, and PyTorch remembers "y came from w * x"
print("y:", y)
print("does y know how it was computed?", y.grad_fn)


y: tensor(6., grad_fn=<MulBackward0>)
does y know how it was computed? <MulBackward0 object at 0x14416ab90>


In [62]:
y.backward()                # compute dy/dw
print("dy/dw:", w.grad)     # should be 3.0 -- since y = w * x, dy/dw = x = 3.0


dy/dw: tensor(3.)


**Try it, predict then check**: if `z = w**2 * x`, what is `dz/dw` at
`w=2, x=3`? Work it out by hand first, then build `z` with autograd,
call `.backward()`, and check.


In [64]:
# TODO: build z = w**2 * x with autograd tracking on w (w=2, x=3), then
# compute and print dz/dw



z: tensor(12., grad_fn=<MulBackward0>)
dz/dw None


/var/folders/16/zvv3j1_10bz2tmjfb04n175r0000gn/T/ipykernel_82974/3216470100.py:5: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print("dz/dw", z.grad)


That's the entire mechanism. Every training loop today, however many
parameters or features are involved, follows the same pattern: build a
computation graph by computing a loss, call `.backward()`, then read off
`.grad`.

### 7.1 Two things you must do around every update

**`torch.no_grad()` when you update a parameter.** The update itself
(`w -= lr * w.grad`) is an operation on `w`. If autograd is left tracking
it, it tries to fold the update into the same graph, which is not what you
want. `torch.no_grad()` tells PyTorch "don't track what happens in this
block."

**`.grad.zero_()` after every step.** `.backward()` **accumulates**
gradients into `.grad`: it adds to whatever's already there, rather than
replacing it. Forget this, and every epoch's gradient piles on top of the
last one. Training doesn't crash. It just quietly stops improving, with no
error message. This is the single most common silent bug in a from-scratch
PyTorch loop.


In [ ]:
# See both in action on the tiny example above
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)
lr = 0.1

for step in range(3):
    y = w * x
    y.backward()

    with torch.no_grad():        # <-- don't track the update itself
        w -= lr * w.grad

    print(f"step {step}: w={w.item():.3f}, grad_used={w.grad.item():.3f}")

    w.grad.zero_()                # <-- reset, or gradients accumulate


Try commenting out the `w.grad.zero_()` line above and re-running the cell.
The printed `grad_used` keeps growing (3, 6, 9, ...) instead of staying at
3.0 every time, because each `.backward()` call adds to whatever was
already sitting in `.grad`. Put the line back before continuing.


## 8. Warm-up: one input feature

Before adding multiple features, let's fit the simplest possible model:
one input, one output. That way the training loop pattern is completely
clear before it becomes your job to write it yourself, in Section 9.

We generate the data ourselves, from a known relationship, specifically so
we can check afterward whether training recovered it.


In [ ]:
torch.manual_seed(0)
rng = np.random.default_rng(0)

n_samples = 200
true_w_single = 3.0
true_b_single = 1.5

x = rng.uniform(-3, 3, size=n_samples)
noise = rng.normal(0, 1.0, size=n_samples)
y = true_w_single * x + true_b_single + noise

x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

plt.scatter(x, y, alpha=0.6)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Synthetic data: true relationship y = {true_w_single}x + {true_b_single} + noise")
plt.show()


In [ ]:
w_single = torch.zeros(1, requires_grad=True)
b_single = torch.zeros(1, requires_grad=True)

def forward_single(x):
    return w_single * x + b_single

def mse_loss(y_pred, y):
    return ((y_pred - y) ** 2).mean()

lr = 0.05
n_epochs = 200
loss_history = []

for epoch in range(n_epochs):
    y_pred = forward_single(x)
    loss = mse_loss(y_pred, y)

    loss.backward()

    with torch.no_grad():
        w_single -= lr * w_single.grad
        b_single -= lr * b_single.grad

    w_single.grad.zero_()
    b_single.grad.zero_()

    loss_history.append(loss.item())
    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}  w={w_single.item():.3f}  b={b_single.item():.3f}")

print(f"\nlearned: w={w_single.item():.3f}, b={b_single.item():.3f}  (true: w={true_w_single}, b={true_b_single})")


In [ ]:
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Loss curve")
plt.show()


The learned `w_single` and `b_single` should land close to the true
values, though not exactly: we added noise on purpose. That gap between
"learned" and "true" is exactly what a validation or test set is for in a
real problem, where you don't get to know the true relationship in
advance.

**Try it**: retrain from scratch (reset the parameters to zero each time)
with `lr = 0.001`, and separately with `lr = 5.0`. Print the final loss and
parameters for each, and note what happens in each case.


In [ ]:
# TODO: retrain from scratch with lr=0.001, then again with lr=5.0. Print
# the final loss and parameters for each, and comment on what you observe.
raise NotImplementedError


## 9. Exercise: multiple input features, and classification

Real problems almost never have just one input. The good news is that
almost nothing about the training loop changes. What changes is the
**shape** of the model:

- `X`: shape `(n_samples, n_features)`. One row per sample, one column per
  feature: this is exactly the shape a pandas DataFrame's numeric columns
  give you.
- `w`: shape `(n_features,)`. One weight per feature, learned.
- `b`: still a single scalar.
- **Forward pass**: `y_pred = X @ w + b`. The matrix multiplication from
  Section 4 combines all features into one number per sample, in a single
  operation, with no loop over features required.

Everything else, the loss, `.backward()`, `torch.no_grad()`, and
`.zero_()`, is identical to Section 8. This section is on you: the data is
given, you write the model and the training loop.

### 9.1 Multi-input linear regression (your turn)


In [ ]:
torch.manual_seed(0)
rng = np.random.default_rng(1)

n_samples = 300
n_features = 3
true_w = np.array([2.0, -1.0, 0.5])
true_b = 4.0

X = rng.uniform(-3, 3, size=(n_samples, n_features))
noise = rng.normal(0, 1.0, size=n_samples)
y = X @ true_w + true_b + noise

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

print("X shape:", X.shape)   # (300, 3)
print("y shape:", y.shape)   # (300,)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for i in range(n_features):
    axes[i].scatter(X[:, i], y, alpha=0.3, s=10)
    axes[i].set_xlabel(f"feature {i}")
    axes[i].set_ylabel("y")
plt.suptitle("y against each feature individually -- no single one tells the whole story")
plt.tight_layout()
plt.show()


Notice that none of the three scatter plots alone looks like a clean line.
The true relationship depends on **all three features together**. That's
exactly why we need a model that combines them, rather than fitting each
feature separately.

**Your turn**: using `X`, `y`, and `n_features` above, and reusing
`mse_loss` from Section 8, write the model and training loop yourself,
generalizing the same four-step pattern from Section 8 (forward pass, loss,
backward, update). You'll need:

- a trainable weight tensor named `w` (one entry per feature) and a
  trainable bias tensor named `b`,
- a `forward` function that takes `X` and returns one prediction per
  sample,
- a training loop that stores each epoch's loss in `loss_history_multi`.

When you're done, `w` should end up close to `[2.0, -1.0, 0.5]` and `b`
close to `4.0`.


In [ ]:
# TODO: define trainable w and b, a forward(X) function, and a training
# loop that stores each epoch's loss in loss_history_multi (see the
# instructions above)
raise NotImplementedError


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(loss_history_multi)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("Loss curve")

with torch.no_grad():
    y_pred = forward(X)
axes[1].scatter(y, y_pred, alpha=0.3, s=10)
lims = [y.min().item(), y.max().item()]
axes[1].plot(lims, lims, color="red", linestyle="--", label="perfect prediction")
axes[1].set_xlabel("true y")
axes[1].set_ylabel("predicted y")
axes[1].set_title("Predicted vs. true")
axes[1].legend()

plt.tight_layout()
plt.show()


If training worked, the right-hand plot should hug the diagonal line. This
is the same check you'll want in Assignment 1. The difference there is
that you won't know the "true" parameters, so a naive baseline (predict the
mean) is what you compare against instead. See the assignment scaffold's
`naive_baseline_score` for that.

### 9.2 Multi-input logistic regression (your turn)

Classification uses the same shapes and the same training loop, changed in
exactly two places: the forward pass ends with a **sigmoid** (squashing the
output into `(0, 1)`, a probability), and the loss becomes **binary
cross-entropy** instead of MSE.


In [ ]:
rng2 = np.random.default_rng(2)

n_samples_clf = 250
n_features_clf = 2
true_w_clf = np.array([1.2, -0.8])
true_b_clf = 0.0

X_clf = rng2.uniform(-3, 3, size=(n_samples_clf, n_features_clf))
z = X_clf @ true_w_clf + true_b_clf
prob_class1 = 1 / (1 + np.exp(-z))
y_clf = rng2.binomial(1, prob_class1)

X_clf = torch.tensor(X_clf, dtype=torch.float32)
y_clf = torch.tensor(y_clf, dtype=torch.float32)

plt.scatter(X_clf[:, 0], X_clf[:, 1], c=y_clf, cmap="coolwarm", alpha=0.7, edgecolors="k", linewidths=0.3)
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.title("Synthetic binary classification data (color = class)")
plt.show()


**Your turn**: using `X_clf`, `y_clf`, and `n_features_clf` above, write the
model and training loop yourself, generalizing 9.1's pattern to
classification. You'll need:

- a trainable weight tensor `w_clf` and bias `b_clf`,
- a `forward_clf` function that turns the linear combination into a
  probability (squash it into `(0, 1)`),
- a `bce_loss` function implementing binary cross-entropy: for a true label
  `y` and predicted probability `y_pred`, the per-sample loss is
  `-(y * log(y_pred) + (1 - y) * log(1 - y_pred))`, averaged over the
  batch. Clamp `y_pred` away from exactly 0 or 1 first, so you never take
  `log(0)`.
- a training loop, same shape as 9.1's, storing each epoch's loss in
  `clf_loss_history`.


In [ ]:
# TODO: define trainable w_clf and b_clf, a forward_clf(X) function, a
# bce_loss(y_pred, y) function, and a training loop that stores each
# epoch's loss in clf_loss_history (see the instructions above)
raise NotImplementedError


In [ ]:
with torch.no_grad():
    y_pred_clf = forward_clf(X_clf)
    pred_labels = (y_pred_clf >= 0.5).float()
    accuracy = (pred_labels == y_clf).float().mean().item()

print(f"accuracy on the training data: {accuracy:.3f}")

plt.plot(clf_loss_history)
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("Loss curve")
plt.show()


If training worked, accuracy should be well above 50% (guessing), and
`w_clf`/`b_clf` should be in the same ballpark as `true_w_clf`/`true_b_clf`.

**Further exploration**, if you have time: change `true_w`, `true_b`, or
`n_features` in 9.1 and re-run. Does training still recover them? What
happens to the loss curve if you set `lr = 2.0`? That's the same
instability you'd see with too high a learning rate in Assignment 1.


## 10. A trained model is not a product

A model sitting in a notebook creates zero business value, because nobody
outside the room can use it. The smallest honest fix is putting a form in
front of it. **Gradio** does exactly that: `gr.Interface` maps input
widgets to a Python function to output widgets.

This wraps the single-input model from Section 8, since it's guaranteed to
be trained regardless of how Section 9 went.


In [ ]:
import gradio as gr

def predict(x_value):
    x_input = torch.tensor([x_value], dtype=torch.float32)
    with torch.no_grad():
        return forward_single(x_input).item()

demo = gr.Interface(
    fn=predict,
    inputs=gr.Number(label="x", value=0.0),
    outputs=gr.Number(label="predicted y"),
    title="Linear regression, from scratch",
    description=(
        f"Model trained by hand (no torch.optim) on synthetic data. "
        f"True relationship: y = {true_w_single}x + {true_b_single}."
    ),
)


Run the cell below to launch it inline in this notebook.
`prevent_thread_lock=True` keeps the kernel responsive, so you can keep
running cells afterward. A real script instead uses
`if __name__ == "__main__":` for the same reason, as you'll see in the
Assignment 1 scaffold. Try values inside the training range, roughly -3 to
3, and then try values far outside it.


In [ ]:
demo.launch(prevent_thread_lock=True)


## Where this goes next

- **Assignment 1**: the same shapes and the same mental model (`X @ w + b`,
  a loss, a training loop), but using `torch.optim` instead, as shown in
  the appendix below.
- **Classification** is also fair game for Assignment 1, using the same
  `torch.optim` pattern applied to the sigmoid/BCE version from Section
  9.2.


## Appendix: the standard way, with `torch.optim`

Everything above wrote the update rule by hand:
`torch.no_grad(): w -= lr * w.grad`, then `w.grad.zero_()`. That's the
mechanism worth understanding once, but nobody writes it that way day to
day. The standard PyTorch workflow uses `torch.optim` for the update and
`torch.nn` for the model, and this is what Assignment 1 expects.

What changes:

- The model becomes an `nn.Module`. For a linear model, that's
  `nn.Linear(n_features, 1)` instead of raw `w`/`b` tensors: it holds both
  the weights and the bias, initialized for you.
- An **optimizer** (`torch.optim.SGD` or `torch.optim.Adam`) is created
  once, pointed at `model.parameters()`.
- Each step becomes: `optimizer.zero_grad()` (replaces `.zero_()` on every
  parameter by hand), forward pass, loss, `loss.backward()`,
  `optimizer.step()` (replaces the manual `torch.no_grad()` update block).

### A.1 Multi-input linear regression, the standard way

Reusing the exact same `X`, `y`, `true_w`, `true_b` from Section 9.1.


In [ ]:
import torch.nn as nn

torch.manual_seed(0)
model_reg = nn.Linear(n_features, 1)
optimizer_reg = torch.optim.Adam(model_reg.parameters(), lr=0.1)
loss_fn_reg = nn.MSELoss()

reg_loss_history = []
for epoch in range(300):
    optimizer_reg.zero_grad()
    y_pred = model_reg(X).squeeze(-1)
    loss = loss_fn_reg(y_pred, y)
    loss.backward()
    optimizer_reg.step()

    reg_loss_history.append(loss.item())
    if epoch % 60 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

learned_w = model_reg.weight.detach().numpy().round(3)
learned_b = model_reg.bias.item()
print(f"\nlearned w: {learned_w}   (true: {true_w})")
print(f"learned b: {learned_b:.3f}          (true: {true_b})")


Compare this loop to Section 9.1's: no `torch.no_grad()` block, no manual
`.zero_()` calls on individual tensors, no raw `w`/`b`. `optimizer.step()`
and `optimizer.zero_grad()` are doing exactly what you did by hand earlier,
just without the bookkeeping.

### A.2 Logistic regression, the standard way

Reusing `X_clf`, `y_clf` from Section 9.2. The model adds a sigmoid after
the linear layer; the loss switches to `nn.BCELoss`.


In [ ]:
torch.manual_seed(0)
model_clf = nn.Sequential(
    nn.Linear(n_features_clf, 1),
    nn.Sigmoid(),
)
optimizer_clf = torch.optim.Adam(model_clf.parameters(), lr=0.1)
loss_fn_clf = nn.BCELoss()

clf_loss_history_optim = []
for epoch in range(300):
    optimizer_clf.zero_grad()
    y_pred = model_clf(X_clf).squeeze(-1)
    loss = loss_fn_clf(y_pred, y_clf)
    loss.backward()
    optimizer_clf.step()

    clf_loss_history_optim.append(loss.item())
    if epoch % 60 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

with torch.no_grad():
    pred_labels = (model_clf(X_clf).squeeze(-1) >= 0.5).float()
    accuracy = (pred_labels == y_clf).float().mean().item()
print(f"\naccuracy on the training data: {accuracy:.3f}")


This is exactly the pattern the Assignment 1 scaffold's `train_template.py`
asks you to implement: `build_model` returns an `nn.Module` like
`model_reg` or `model_clf` above, and `train_model` runs a loop like the
two above, on your own dataset instead of synthetic data.
